# ML-07 — Baseline Action Score and Top-10 Review

**Lane locked for Week 4: CTR / Engagement Opportunity Scoring, CTR scope.** This continues ML-02 and ML-03: help an editor prioritize visible pages that capture fewer clicks than position/content-type peers. Engagement is outside this baseline.

Prepared with AI assistance. This is an observed, same-snapshot decision-support queue, not a forecast of future CTR or proof that an edit increases clicks. All evidence below comes from the approved local starter slice. No warehouse access is needed.

**Reproduce:** install `pandas`, `numpy`, `nbformat`, `nbclient`, and `ipykernel` in a Python environment and run all cells from the repository, starter, or notebook folder. The notebook regenerates `work/outputs/baseline_action_score.csv` (gitignored) and `work/outputs/w04_baseline_metrics.json` (committed receipt). The shared pipeline is unchanged.

References inside the starter folder: `docs/data-dictionary.md`, `DATA_USE.md`, `skills/building-baselines/SKILL.md`, and `skills/flyrank/flyrank-data/SKILL.md`.

## 1) My rule, two signal checks, and reasoning

**Rule in plain words:** review pages with at least 500 trailing-90-day impressions and recorded average position 1–20. Compare exact CTR with the pooled CTR of pages in the same position band and content type, excluding the candidate's entire client and requiring at least 30 peer pages from 3 other clients. Rank positive gaps by `impressions × (peer CTR − observed CTR) / 100`, with one reason code **`below_position_type_peer_ctr`** and one action **`review_snippet_and_intent`**.

These are the policies proposed in ML-03; no weights or thresholds are fitted here. The score has click-count units but is a **review-priority proxy**, not expected recoverable clicks. Pages without enough volume, recorded position, or peer support abstain; supported pages without a positive gap are not recommended.

Before encoding the rule, I test two hypotheses. (A) Better position has higher measured CTR, the signal behind FlyRank's **CTR-fix** logic. (B) Greater impression volume makes observed CTR less coarse and zero clicks less common, supporting the volume gate and the demand idea behind **quick-win**. Neither test uses a decline label or a future window. Each verdict describes only this starter snapshot.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

relative_csv = Path("data/raw/content_refresh_anonymized.csv")
root = next(
    (candidate for base in (Path.cwd(), *Path.cwd().parents)
     for candidate in (base, base / "flyrank-ml-internship-starter-main")
     if (candidate / relative_csv).is_file()), None
)
if root is None:
    raise FileNotFoundError("Run inside a clone containing the approved starter CSV.")

# Read an explicit allowlist. Label, trend, recent-window and product flags never enter df.
GROUP_KEYS = ["content_id", "client_id"]
SCORE_INPUTS = ["content_type", "avg_position", "impressions_90d", "clicks_90d"]
df = pd.read_csv(root / relative_csv, usecols=GROUP_KEYS + SCORE_INPUTS)
assert df[GROUP_KEYS].notna().all().all()
assert df["content_id"].is_unique, "Expected one row per content item."
assert df[SCORE_INPUTS].notna().all().all()
assert df[["impressions_90d", "clicks_90d"]].ge(0).all().all()
assert df["impressions_90d"].gt(0).all()
assert df["clicks_90d"].le(df["impressions_90d"]).all()

MIN_IMPRESSIONS, MIN_PEER_PAGES, MIN_PEER_CLIENTS, K = 500, 30, 3, 20
POSITION_EDGES = [0, 3, 10, 20]
POSITION_LABELS = ["1–3", ">3–10", ">10–20"]
lane = df.loc[df["impressions_90d"].ge(MIN_IMPRESSIONS)
              & df["avg_position"].between(1, 20)].copy()
lane["source_row"] = lane.index  # Final tie-break only; not a scoring feature.
lane["observed_ctr_pct"] = 100 * lane["clicks_90d"] / lane["impressions_90d"]
lane["position_band"] = pd.cut(lane["avg_position"], POSITION_EDGES,
                              labels=POSITION_LABELS)
assert not lane.empty
print(f"Starter: {len(df):,} pages, {df['client_id'].nunique()} clients; one row per page.")
print(f"Lane: {len(lane):,} pages across {lane['client_id'].nunique()} clients.")
print(f"Position 0 (missing) excluded: {df['avg_position'].eq(0).sum():,} pages.")
print("Time contract: trailing 90 days ending at export; calendar export date not supplied.")
print("CTR units: percentage points; recomputed from counts to avoid stored-rate rounding.")

Starter: 30,000 pages, 32 clients; one row per page.
Lane: 12,009 pages across 28 clients.
Position 0 (missing) excluded: 1,205 pages.
Time contract: trailing 90 days ending at export; calendar export date not supplied.
CTR units: percentage points; recomputed from counts to avoid stored-rate rounding.


### Signal A — position versus CTR (flag-linked)

Hypothesis: among pages with ≥500 impressions, pooled CTR decreases as position worsens. The buckets use the same position bands planned for the rule. Print **n**, client count, exposure, pooled CTR and median page CTR; pooled CTR is `100 × sum(clicks) / sum(impressions)`, not the mean of rounded percentages. Content-type context follows so a sparse category cannot masquerade as a reliable peer group.

In [2]:
position_table = lane.groupby("position_band", observed=True).agg(
    n=("content_id", "size"), clients=("client_id", "nunique"),
    impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum"),
    median_page_ctr_pct=("observed_ctr_pct", "median")
)
position_table["pooled_ctr_pct"] = 100 * position_table["clicks"] / position_table["impressions"]
assert position_table["n"].sum() == len(lane)
display(position_table.round(4))

type_position_table = lane.groupby(["content_type", "position_band"], observed=True).agg(
    n=("content_id", "size"), clients=("client_id", "nunique"),
    impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum")
)
type_position_table["pooled_ctr_pct"] = (
    100 * type_position_table["clicks"] / type_position_table["impressions"]
)
display(type_position_table.round(4))
SIGNAL_A_VERDICT = "MIXED"
print("Signal A verdict:", SIGNAL_A_VERDICT)

,n,clients,impressions,clicks,median_page_ctr_pct,pooled_ctr_pct
position_band,,,,,,
1–3,466,17,7436196,36784,0.2297,0.4947
>3–10,7084,28,88871350,309691,0.2368,0.3485
>10–20,4459,25,22384996,78544,0.1734,0.3509


n  clients  impressions  clicks  \
content_type       position_band                                       
comparison article >3–10            50        1        80067      34   
                   >10–20           11        1        10979      17   
feedly article     1–3               3        2         9416     233   
                   >3–10            83        6       214524     929   
                   >10–20           34        3        75191     172   
keyword article    1–3             463       15      7426780   36551   
                   >3–10          6951       28     88576759  308728   
                   >10–20         4414       25     22298826   78355   

                                  pooled_ctr_pct  
content_type       position_band                  
comparison article >3–10                  0.0425  
                   >10–20                 0.1548  
feedly article     1–3                    2.4745  
                   >3–10                  0.4331  
                   >10–20                 0.2288  
keyword article    1–3                    0.4922  
                   >3–10                  0.3485  
                   >10–20                 0.3514

Signal A verdict: MIXED


**MIXED.** Pooled CTR is about **0.4947%, 0.3485%, 0.3509%** across the three bands: the best-position band has higher CTR, but the latter two reverse slightly. Median page CTR is also non-monotonic (**0.2297%, 0.2368%, 0.1734%**). This saves the rule from assuming a universally decreasing CTR curve or using one global 0.5% cutoff. I retain empirical position/type comparisons and report their limits. The top-position feedly group has only **n=3**; comparison articles span only one client. Position is an average over mixed queries/devices, and these descriptive tables do not isolate a causal position effect.

### Signal B — impression volume and measurement resolution (flag-linked)

Hypothesis: low exposure yields coarser CTR and more zero-click pages. Keep valid positions 1–20, then bucket impressions including pages below the proposed gate. A single click changes CTR by `100 / impressions` percentage points. The median Wilson 95% interval width provides an additional uncertainty diagnostic; its binomial assumptions ignore repeated users and query clustering, so it is illustrative, not calibrated editorial confidence.

In [3]:
volume_view = df.loc[df["avg_position"].between(1, 20)].copy()
volume_view["volume_band"] = pd.cut(
    volume_view["impressions_90d"], [0, 99, 499, 999, 4999, np.inf],
    labels=["1–99", "100–499", "500–999", "1000–4999", "5000+"]
)
volume_view["one_click_step_pp"] = 100 / volume_view["impressions_90d"]
volume_view["zero_click"] = volume_view["clicks_90d"].eq(0)
n = volume_view["impressions_90d"].to_numpy(dtype=float)
p = volume_view["clicks_90d"].to_numpy(dtype=float) / n
z = 1.959963984540054
volume_view["wilson_width_pp"] = (
    200 * z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
)
volume_table = volume_view.groupby("volume_band", observed=True).agg(
    n=("content_id", "size"), clients=("client_id", "nunique"),
    median_one_click_step_pp=("one_click_step_pp", "median"),
    max_one_click_step_pp=("one_click_step_pp", "max"),
    zero_click_pct=("zero_click", lambda values: 100 * values.mean()),
    median_wilson_width_pp=("wilson_width_pp", "median")
)
assert volume_table["n"].sum() == len(volume_view)
display(volume_table.round(4))
SIGNAL_B_VERDICT = "CONFIRMED"
print("Signal B verdict:", SIGNAL_B_VERDICT)

,n,clients,median_one_click_step_pp,max_one_click_step_pp,zero_click_pct,median_wilson_width_pp
volume_band,,,,,,
1–99,5094,30,6.6667,100.00,86.0031,21.5311
100–499,3061,28,0.4000,1.00,59.9151,1.9032
500–999,2010,28,0.1403,0.20,32.7861,0.7730
1000–4999,5305,27,0.0446,0.10,9.8398,0.4041
5000+,4694,24,0.0084,0.02,0.6817,0.1744


Signal B verdict: CONFIRMED


**CONFIRMED** for measurement resolution and the observed zero-click pattern, not for actionable edits. The **1–99** bucket has **n=5,094**, a median single-click step of **6.6667 pp**, and **86.00%** zero-click pages. At **500–999**, **n=2,010**, these are **0.1403 pp** and **32.79%**; at **5000+**, **n=4,694**, **0.0084 pp** and **0.68%**. Thus a zero-click page with tiny exposure is weak evidence for a CTR problem. The 500 gate is a provisional floor and does not eliminate noise: at exactly 500, one click still moves CTR by **0.2 pp**. Bucket composition, position and client mix also change, so higher volume is not shown to cause higher CTR. Volume prioritizes reach and measurement support; it is not an editorial target.

## 2) Build the ranked queue (writes the CSV)

One frozen rule, no training. The benchmark excludes the entire candidate client rather than just the candidate page. Unsupported benchmarks stay missing. The CSV contains only positively scored review candidates, ordered by score, then impressions, then stable source row; IDs link the queue to local records and group peers, never contribute to the score. Each row has exactly one reason code and action label.

In [4]:
peer_keys = ["position_band", "content_type"]
grouped = lane.groupby(peer_keys, observed=True)
client_grouped = lane.groupby(peer_keys + ["client_id"], observed=True)
other_impressions = (grouped["impressions_90d"].transform("sum")
                     - client_grouped["impressions_90d"].transform("sum"))
other_clicks = (grouped["clicks_90d"].transform("sum")
                - client_grouped["clicks_90d"].transform("sum"))
lane["peer_pages_other_clients"] = (grouped["content_id"].transform("size")
                                     - client_grouped["content_id"].transform("size"))
lane["peer_clients_other"] = grouped["client_id"].transform("nunique") - 1
supported = (lane["peer_pages_other_clients"].ge(MIN_PEER_PAGES)
             & lane["peer_clients_other"].ge(MIN_PEER_CLIENTS)
             & other_impressions.gt(0))
lane["peer_ctr_pct"] = (100 * other_clicks / other_impressions).where(supported)
lane["ctr_gap_pp"] = (lane["peer_ctr_pct"] - lane["observed_ctr_pct"]).clip(lower=0)
lane["baseline_action_score"] = lane["impressions_90d"] * lane["ctr_gap_pp"] / 100
assert lane.loc[~supported, "baseline_action_score"].isna().all()

queue = lane.loc[supported & lane["baseline_action_score"].gt(0)].sort_values(
    ["baseline_action_score", "impressions_90d", "source_row"],
    ascending=[False, False, True], kind="stable"
).copy()
queue["baseline_rank"] = np.arange(1, len(queue) + 1)
queue["reason_code"] = "below_position_type_peer_ctr"
queue["action_label"] = "review_snippet_and_intent"
queue_columns = ["content_id", "client_id", "baseline_rank", "baseline_action_score",
                 "reason_code", "action_label", "content_type", "position_band",
                 "impressions_90d", "clicks_90d", "avg_position", "observed_ctr_pct",
                 "peer_ctr_pct", "ctr_gap_pp", "peer_pages_other_clients", "peer_clients_other"]
output_dir = root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "baseline_action_score.csv"
queue[queue_columns].to_csv(csv_path, index=False)

support_table = lane.assign(supported=supported).groupby("content_type", observed=True).agg(
    n=("content_id", "size"), supported_pages=("supported", "sum")
)
support_table["unsupported_pages"] = support_table["n"] - support_table["supported_pages"]
display(support_table)
print(f"Supported comparisons: {supported.sum():,}; unsupported: {(~supported).sum():,}.")
print(f"Positive-gap queue: {len(queue):,}; supported without a positive gap: {(supported & lane['baseline_action_score'].eq(0)).sum():,}.")
print("Wrote work/outputs/baseline_action_score.csv (regenerated locally; do not commit).")

,n,supported_pages,unsupported_pages
content_type,,,
comparison article,61,0,61
feedly article,120,83,37
keyword article,11828,11828,0


Supported comparisons: 11,911; unsupported: 98.
Positive-gap queue: 8,429; supported without a positive gap: 3,482.
Wrote work/outputs/baseline_action_score.csv (regenerated locally; do not commit).


### Evaluation status at K=20

Compare the peer-gap rule to an impressions-only baseline on the same supported pool and review budget. Report operational coverage, overlap, and client concentration. Independent editorial `review_worth_action` labels are absent, so **precision@20 and its base rate are unavailable for both rankings**. I do not substitute the decline label (wrong lane) or define truth from the rule itself. The ten reviews below are evidence critiques, not actual snippet/intent judgments: the export has no snippets, URLs or queries.

In [5]:
impressions_queue = lane.loc[supported].sort_values(
    ["impressions_90d", "source_row"], ascending=[False, True], kind="stable"
)
evaluation_status = pd.DataFrame([
    {"ranking": name, "K": K, "queue_coverage_at_K": min(len(ranking), K) / K,
     "top_K_clients": ranking.head(K)["client_id"].nunique(),
     "independent_review_labels": 0, "precision_at_20": None, "editorial_base_rate": None}
    for name, ranking in [("peer-gap rule", queue), ("impressions-only", impressions_queue)]
])
top_k_overlap = len(set(queue.head(K)["content_id"]) & set(impressions_queue.head(K)["content_id"]))
display(evaluation_status)
print(f"Top-20 overlap with impressions-only: {top_k_overlap}/{K}.")
print("Precision@20 / editorial base rate: unavailable; no independent labels.")

,ranking,K,queue_coverage_at_K,top_K_clients,independent_review_labels,precision_at_20,editorial_base_rate
0,peer-gap rule,20,1.0,5,0,None,None
1,impressions-only,20,1.0,7,0,None,None


Top-20 overlap with impressions-only: 10/20.
Precision@20 / editorial base rate: unavailable; no independent labels.


## 3) Top-10 review

Each line links the rank to its CSV page, states the action and observed reason, and says what would invalidate it. Confidence is a qualitative caution about the proxy, not a probability of action success. First inspect snippet, intent, query/device mix, and count validity in an authorized workflow; only then decide whether an edit is justified. No page has been edited here.

In [6]:
# Inspect peer concentration without publishing client IDs or a row-level data dump.
top10 = queue.head(10).copy()
peer_diagnostics = []
for _, row in top10.iterrows():
    peers = lane.loc[lane["position_band"].eq(row["position_band"])
                     & lane["content_type"].eq(row["content_type"])
                     & lane["client_id"].ne(row["client_id"])]
    by_client = peers.groupby("client_id").agg(
        impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum")
    )
    client_ctrs = 100 * by_client["clicks"] / by_client["impressions"]
    # Independent direct aggregation confirms whole-client exclusion.
    assert np.isclose(row["peer_ctr_pct"], 100 * peers["clicks_90d"].sum() / peers["impressions_90d"].sum())
    peer_diagnostics.append({
        "baseline_rank": int(row["baseline_rank"]),
        "largest_peer_client_impression_share": float(by_client["impressions"].max() / by_client["impressions"].sum()),
        "median_peer_client_ctr_pct": float(client_ctrs.median())
    })
top10 = top10.merge(pd.DataFrame(peer_diagnostics), on="baseline_rank", validate="one_to_one")

# Ten individual critiques of the observed picks; no independent editorial labels implied.
invalidators = [
    "One peer client supplies about 75% of benchmark impressions; a matched query/device mix could erase the apparent gap. Median-client CTR is much closer than the pooled benchmark.",
    "Even at position 2.3, zero-click SERP features or mismatched intent could explain the gap; a title/meta edit would then not address its cause.",
    "Position 7.3 sits in a broad >3–10 band; comparable peers near position 7 could have lower CTR than the pooled band and change the priority.",
    "About 63% of peer impressions come from one client; if that client has different branded demand, the benchmark may overstate the deficit.",
    "The gap is only about 0.18 pp; high volume drives its rank. If continuous-position peers at 5.4 have this CTR naturally, an edit is unjustified.",
    "Zero clicks despite 208,678 impressions needs a reporting/query-intent check first; measurement mismatch or zero-click demand would invalidate an editorial interpretation.",
    "Position 5.6 hides query/device variation; a narrow, matched peer comparison or an already-appropriate snippet could remove the case for action.",
    "Position 4.0 is just beyond the band boundary at 3; small shifts in the recorded average or band definition could change the peer benchmark and rank.",
    "Position 7.8 plus a peer client contributing about 57% of impressions creates both within-band and client-mix risk; matched context may explain its low CTR.",
    "Position 2.9 is near the band edge and about 75% of peer impressions come from one client; narrower bands or client-balanced peers could reduce its priority."
]
confidences = ["low", "medium", "medium", "low", "low", "low", "medium", "low", "medium", "low"]
assert len(top10) == len(invalidators) == 10
review_lines = []
for i, row in top10.iterrows():
    line = (
        f"**{int(row['baseline_rank'])}. `{row['content_id']}` — {row['action_label']}** "
        f"(`{row['reason_code']}`): {int(row['impressions_90d']):,} impressions, position {row['avg_position']:.1f}, "
        f"CTR {row['observed_ctr_pct']:.4f}% vs peers {row['peer_ctr_pct']:.4f}%, "
        f"score {row['baseline_action_score']:.2f}. Proxy confidence: **{confidences[i]}**. "
        f"**What would make it wrong:** {invalidators[i]}"
    )
    review_lines.append(line)
display(Markdown("\n\n".join(review_lines)))
print(f"Top-10 review covers 10 distinct pages across {top10['client_id'].nunique()} clients.")

**1. `content_8c19996aa890` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 509,252 impressions, position 2.5, CTR 0.1541% vs peers 0.5327%, score 1927.56. Proxy confidence: **low**. **What would make it wrong:** One peer client supplies about 75% of benchmark impressions; a matched query/device mix could erase the apparent gap. Median-client CTR is much closer than the pooled benchmark.

**2. `content_8451fc6f034d` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 272,144 impressions, position 2.3, CTR 0.0276% vs peers 0.5099%, score 1312.67. Proxy confidence: **medium**. **What would make it wrong:** Even at position 2.3, zero-click SERP features or mismatched intent could explain the gap; a title/meta edit would then not address its cause.

**3. `content_36ff89c8214e` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 295,097 impressions, position 7.3, CTR 0.0522% vs peers 0.4228%, score 1093.56. Proxy confidence: **medium**. **What would make it wrong:** Position 7.3 sits in a broad >3–10 band; comparable peers near position 7 could have lower CTR than the pooled band and change the priority.

**4. `content_5fe46e04994d` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 517,715 impressions, position 4.2, CTR 0.1431% vs peers 0.3502%, score 1072.07. Proxy confidence: **low**. **What would make it wrong:** About 63% of peer impressions come from one client; if that client has different branded demand, the benchmark may overstate the deficit.

**5. `content_aaef01a50def` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 517,109 impressions, position 5.4, CTR 0.2456% vs peers 0.4228%, score 916.14. Proxy confidence: **low**. **What would make it wrong:** The gap is only about 0.18 pp; high volume drives its rank. If continuous-position peers at 5.4 have this CTR naturally, an edit is unjustified.

**6. `content_c8e9d6ab9013` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 208,678 impressions, position 9.7, CTR 0.0000% vs peers 0.4228%, score 882.21. Proxy confidence: **low**. **What would make it wrong:** Zero clicks despite 208,678 impressions needs a reporting/query-intent check first; measurement mismatch or zero-click demand would invalidate an editorial interpretation.

**7. `content_cb112fce36be` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 309,910 impressions, position 5.6, CTR 0.1588% vs peers 0.4228%, score 818.18. Proxy confidence: **medium**. **What would make it wrong:** Position 5.6 hides query/device variation; a narrow, matched peer comparison or an already-appropriate snippet could remove the case for action.

**8. `content_1a9e894be2e2` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 416,180 impressions, position 4.0, CTR 0.2268% vs peers 0.4228%, score 815.45. Proxy confidence: **low**. **What would make it wrong:** Position 4.0 is just beyond the band boundary at 3; small shifts in the recorded average or band definition could change the peer benchmark and rank.

**9. `content_c84a0ab98e90` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 223,271 impressions, position 7.8, CTR 0.0314% vs peers 0.3541%, score 720.52. Proxy confidence: **medium**. **What would make it wrong:** Position 7.8 plus a peer client contributing about 57% of impressions creates both within-band and client-mix risk; matched context may explain its low CTR.

**10. `content_e12868d1f396` — review_snippet_and_intent** (`below_position_type_peer_ctr`): 149,712 impressions, position 2.9, CTR 0.0695% vs peers 0.5327%, score 693.45. Proxy confidence: **low**. **What would make it wrong:** Position 2.9 is near the band edge and about 75% of peer impressions come from one client; narrower bands or client-balanced peers could reduce its priority.

Top-10 review covers 10 distinct pages across 4 clients.


## 4) Weak picks and leakage check

**Weak picks:** ranks **1 and 10** share a top-position benchmark dominated by one other client (~75% of impressions). Rank **4** has a similarly concentrated benchmark (~63%). Rank **5** gets a large score partly through volume despite a relatively small gap, and rank **6** needs a count-validity check before editorial interpretation. Ranks **8 and 10** sit near band boundaries. These are reasons to inspect or defer, not fabricated negative labels. The top ten cover only four clients, so a global queue can consume the editor's budget on a few clients.

**Benchmark sensitivity:** compute an alternative using the median of other clients' pooled CTRs (equal client weight) and check top-20 overlap. This is a diagnostic, not a second selected rule or a retuned submission. Keep the frozen pooled rule as Week 5's comparison target. More appropriate query/device comparisons and independent reviews remain future work.

In [7]:
client_totals = lane.groupby(peer_keys + ["client_id"], observed=True).agg(
    impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum")
).reset_index()
client_totals["client_ctr_pct"] = 100 * client_totals["clicks"] / client_totals["impressions"]
median_lookup = {}
for key, group in client_totals.groupby(peer_keys, observed=True):
    for client in group["client_id"]:
        median_lookup[(*key, client)] = group.loc[group["client_id"].ne(client), "client_ctr_pct"].median()
lane["client_balanced_peer_ctr_pct"] = [
    median_lookup[(band, kind, client)]
    for band, kind, client in zip(lane["position_band"], lane["content_type"], lane["client_id"])
]
lane["client_balanced_diagnostic_score"] = (
    lane["impressions_90d"]
    * (lane["client_balanced_peer_ctr_pct"] - lane["observed_ctr_pct"]).clip(lower=0) / 100
).where(supported)
alternative = lane.loc[supported & lane["client_balanced_diagnostic_score"].gt(0)].sort_values(
    ["client_balanced_diagnostic_score", "impressions_90d", "source_row"],
    ascending=[False, False, True], kind="stable"
)
balanced_overlap = len(set(queue.head(K)["content_id"]) & set(alternative.head(K)["content_id"]))
display(top10[["baseline_rank", "largest_peer_client_impression_share", "median_peer_client_ctr_pct"]].round(4))
print(f"Equal-client benchmark diagnostic: {balanced_overlap}/{K} top-20 pages overlap the frozen rule.")
print("This measures ranking sensitivity, not precision or improved actionability.")

,baseline_rank,largest_peer_client_impression_share,median_peer_client_ctr_pct
0,1,0.7481,0.2053
1,2,0.5798,0.2715
2,3,0.3050,0.3918
3,4,0.6323,0.3918
4,5,0.3050,0.3918
5,6,0.3050,0.3918
6,7,0.3050,0.3918
7,8,0.3050,0.3918
8,9,0.5654,0.3918
9,10,0.7481,0.2053


Equal-client benchmark diagnostic: 16/20 top-20 pages overlap the frozen rule.
This measures ranking sensitivity, not precision or improved actionability.


**Leakage/time audit:** the only score inputs are trailing-90-day impressions/clicks, recorded average position, and content type. IDs group peers and link outputs. `trend_pct`, `trend_direction`, `is_declining_label`, product flags/scores, provider/model metadata, and all `*_last_30d`/`*_prev_30d` fields are excluded at load time. No label-derived target is manufactured. Peer tables use the current snapshot; leave-one-client-out benchmarking prevents own-client influence but is **not** a held-out test or future validation.

The feature window and measurement window deliberately coincide for current review triage. A future-CTR model must use earlier features and separate later outcomes; the CSV lacks an export date and daily history, so this notebook cannot establish future performance. Missing content metadata outside the allowlist is neither imputed nor used. The optional median-client diagnostic leaves the primary CSV untouched.

In [8]:
forbidden = {"trend_pct", "trend_direction", "is_declining_label", "health_score",
             "priority_score", "needs_ctr_fix", "is_quick_win", "provider_used", "model_used"}
assert not forbidden.intersection(df.columns)
assert not any("last_30d" in col or "prev_30d" in col for col in df.columns)
assert set(df.columns) == set(GROUP_KEYS + SCORE_INPUTS)
assert queue["reason_code"].eq("below_position_type_peer_ctr").all()
assert queue["action_label"].eq("review_snippet_and_intent").all()
assert queue["baseline_action_score"].gt(0).all()
assert queue["baseline_action_score"].is_monotonic_decreasing
assert queue["content_id"].is_unique
assert queue["peer_pages_other_clients"].ge(MIN_PEER_PAGES).all()
assert queue["peer_clients_other"].ge(MIN_PEER_CLIENTS).all()
assert queue["avg_position"].between(1, 20).all()
assert queue["impressions_90d"].ge(MIN_IMPRESSIONS).all()
saved = pd.read_csv(csv_path)
assert saved["content_id"].tolist() == queue["content_id"].tolist()
assert saved["baseline_rank"].tolist() == list(range(1, len(queue) + 1))
assert np.allclose(saved["baseline_action_score"], queue["baseline_action_score"])
print("PASS: page grain, allowed inputs, peer support, deterministic ranks, and CSV round-trip.")

def table_records(table):
    # JSON-safe aggregates only; no dataset rows or client identifiers in the receipt.
    return json.loads(table.reset_index().to_json(orient="records", double_precision=10))

metrics = {
    "assignment": "ML-07", "lane": "CTR / Engagement Opportunity Scoring (CTR only)",
    "input_sha256": hashlib.sha256((root / relative_csv).read_bytes()).hexdigest(),
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "analysis_type": "same-snapshot descriptive decision-support; no training or future claims",
    "input_rows": len(df), "input_clients": int(df["client_id"].nunique()),
    "score_inputs": SCORE_INPUTS, "grouping_keys": GROUP_KEYS,
    "policy": {"minimum_impressions": MIN_IMPRESSIONS, "position_range": [1, 20],
               "position_edges": POSITION_EDGES, "minimum_peer_pages": MIN_PEER_PAGES,
               "minimum_other_clients": MIN_PEER_CLIENTS, "review_budget_K": K,
               "peer_method": "pooled CTR excluding candidate's entire client"},
    "signal_verdicts": {"position_vs_ctr": SIGNAL_A_VERDICT, "volume_resolution": SIGNAL_B_VERDICT},
    "position_buckets": table_records(position_table), "volume_buckets": table_records(volume_table),
    "eligible_pages": len(lane), "supported_pages": int(supported.sum()),
    "unsupported_pages": int((~supported).sum()), "positive_gap_queue_pages": len(queue),
    "top_10_reviewed_rows": len(review_lines), "top_10_clients": int(top10["client_id"].nunique()),
    "top_20_clients": int(queue.head(K)["client_id"].nunique()),
    "top_20_overlap_impressions_only": top_k_overlap,
    "top_20_overlap_client_balanced_diagnostic": balanced_overlap,
    "independent_editorial_labels": 0, "precision_at_20": None, "editorial_base_rate": None,
    "metric_limitation": "Independent editorial judgments absent; rule outputs are not labels.",
    "output_csv": "work/outputs/baseline_action_score.csv",
    "verification": "allowlist, grain, support, ranking, peer exclusion and CSV round-trip passed"
}
metrics_path = output_dir / "w04_baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
print("Wrote aggregate run receipt: work/outputs/w04_baseline_metrics.json")

PASS: page grain, allowed inputs, peer support, deterministic ranks, and CSV round-trip.
Wrote aggregate run receipt: work/outputs/w04_baseline_metrics.json


## 5) Self-check

- [x] Confirmed the CTR opportunity lane for Week 4, continuing ML-02 and ML-03.
- [x] Two visible signal bucket tables contain n and one-word verdicts: **MIXED**, **CONFIRMED**; both identify their FlyRank flag connection.
- [x] One plain-language rule produces a numeric score, exactly one reason code and one action label.
- [x] Notebook writes the ranked queue to `work/outputs/baseline_action_score.csv`; that data file stays out of Git.
- [x] Ten distinct ranked pages each have an action, observed reason, confidence caution and what would make the pick wrong.
- [x] Identified weak picks and measured peer-concentration / ranking sensitivity.
- [x] No future-window, label-derived, product-flag or ID scoring inputs; rates use percentage units and position 0 is excluded.
- [x] Precision@20 and editorial base rate remain unavailable; no fabricated labels or causal claims.
- [x] All code cells executed top to bottom; grain, support, peer exclusion and CSV round-trip assertions passed.
- [x] Public outputs are aggregates and the ten approved pseudonymized review references; no client names, URLs, titles or raw queries.

**Next evidence needed:** independent editorial judgments under the ML-03 rubric, matched search-result context, and separate time windows before any future-outcome claims. This frozen baseline is the target to beat in Week 5.